# RetinaNet model for snowpole detection with LiDar and RGB datasets

### Import Packages

In [ ]:
import torchvision
import torch
import torch.optim as optim
from torchvision.models.detection import RetinaNet_ResNet50_FPN_V2_Weights
from torchvision.models.detection.retinanet import RetinaNetClassificationHead
import pandas as pd
from torch.utils.data import Dataset
from PIL import Image # Or import cv2 if using OpenCV
import os
from transform import get_transforms, collate_fn
from dataloader import LidarDataset

In [ ]:
!pip install --upgrade "numpy<2"

### 1. Load the pre-trained model (using V2 weights, which are often better)

In [ ]:

weights = RetinaNet_ResNet50_FPN_V2_Weights.DEFAULT # Loads COCO pre-trained weights
model = torchvision.models.detection.retinanet_resnet50_fpn_v2(weights=weights)


### --- Modification for Custom Dataset ---
#### Get the number of input features for the classifier

In [ ]:
num_anchors = model.head.classification_head.num_anchors
in_channels = model.backbone.out_channels

#### Define your number of classes (e.g., number of classes in your custom dataset + 1 for background)
 IMPORTANT: Torchvision RetinaNet usually expects num_classes = actual_classes + 1 (for background)
 However, documentation suggests the head should be built with num_classes = actual_classes * num_anchors.
 Double-check the specific documentation for the version you use, but often it's handled like this:

In [ ]:
num_classes_custom = 2 # Replace with the actual number of classes in your dataset

In [ ]:
new_cls_head = RetinaNetClassificationHead(
    in_channels=in_channels,
    num_anchors=num_anchors,
    num_classes=num_classes_custom
)

In [ ]:
# Replace the pre-trained head with the new one
model.head.classification_head = new_cls_head

# --- End Modification ---

In [ ]:
model.train()##put model in training mode

In [ ]:
from torch.utils.data import DataLoader

In [ ]:
# --- 1. Define Configuration ---
BATCH_SIZE = 8
ANNOTATIONS_DIR = "/work/mathiamt/SnowPoleDetection/torchVision/pytorch-retinanet/annotations"
CLASSES_FILE = "/work/mathiamt/SnowPoleDetection/torchVision/pytorch-retinanet/classes.csv"
# Define image directory - IMPORTANT: adjust this based on paths in your CSV
# If paths in CSV are like 'retinanet_data/lidar/images/train/image_1.png' and you run from project root:
IMG_DIR = '.'
# If paths in CSV are just 'image_1.png', use the full path to the specific train/valid directories
# IMG_DIR_TRAIN = '/work/mathiamt/SnowPoleDetection/torchVision/pytorch-retinanet/retinanet_data/lidar/images/train'
# IMG_DIR_VALID = '/work/mathiamt/SnowPoleDetection/torchVision/pytorch-retinanet/retinanet_data/lidar/images/valid'

# Get the transform functions
train_transforms = get_transforms(is_train=True)
val_transforms = get_transforms(is_train=False)

# --- 2. Create Dataset Instances ---
train_annotations_file = os.path.join(ANNOTATIONS_DIR, "lidar_train_annotations.csv")
val_annotations_file = os.path.join(ANNOTATIONS_DIR, "lidar_valid_annotations.csv")

# Make sure to provide all required arguments to LidarDataset:
# annotations_file, classes_file, img_dir, transforms
train_dataset = LidarDataset(
    annotations_file=train_annotations_file,
    classes_file=CLASSES_FILE,
    img_dir=None, # Or IMG_DIR_TRAIN if paths in CSV are just filenames
    transforms=train_transforms
)

val_dataset = LidarDataset(
    annotations_file=val_annotations_file,
    classes_file=CLASSES_FILE,
    img_dir=None, # Or IMG_DIR_VALID if paths in CSV are just filenames
    transforms=val_transforms
)

# --- 3. Create DataLoader Instances ---
# Now pass the dataset instances to the DataLoader
train_loader = DataLoader(
    dataset=train_dataset, # Pass the instantiated dataset
    batch_size=BATCH_SIZE,
    shuffle=True,          # Shuffle for training
    num_workers=2,         # Adjust based on your system
    collate_fn=collate_fn
)

val_loader = DataLoader(
    dataset=val_dataset,   # Pass the instantiated dataset
    batch_size=BATCH_SIZE,
    shuffle=False,         # No shuffle for validation
    num_workers=2,
    collate_fn=collate_fn
)

# --- Ready for Training ---
print(f"Created train_loader with {len(train_loader)} batches.")
print(f"Created val_loader with {len(val_loader)} batches.")

# You can now use train_loader and val_loader in your training loop

##### --- 1. Define the Model ---

In [ ]:
# --- 2. Set the Device ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
model.to(device) # Move model to the chosen device




In [ ]:
# --- 3. Define the Optimizer ---
# Get parameters that require gradients
params = [p for p in model.parameters() if p.requires_grad]

In [ ]:
# Choose optimizer and learning rate
# Start with Adam or SGD, common values for lr are 0.001, 0.0005, 0.0001
learning_rate = 0.0005
# optimizer = optim.Adam(params, lr=learning_rate, weight_decay=0.0001)
optimizer = optim.SGD(params, lr=learning_rate, momentum=0.9, weight_decay=0.0005) # SGD often used in papers

print("Model, Device, and Optimizer are set up.")

In [ ]:
import time # Optional: for timing epochs

# --- Assuming previous setup is done ---
# model, device, optimizer should be defined
# train_loader, val_loader should be defined
# num_actual_classes should be defined

# --- Training Configuration ---
num_epochs = 10 # Or choose desired number of epochs

# Optional: Learning Rate Scheduler
# Example: Reduce LR by a factor of 0.1 every 3 epochs
# scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=3, gamma=0.1)

# Lists to store losses for plotting later (optional)
train_loss_history = []
# Note: Calculating validation loss requires passing targets in eval mode,
# which torchvision models don't directly support for loss calculation.
# We'll focus on training loss and add validation metric calculation later.

print("Starting Training...")
start_time = time.time()

for epoch in range(num_epochs):
    # --- Training Phase ---
    model.train() # Set model to training mode
    running_loss = 0.0
    epoch_train_loss = 0.0
    print(f"\nEpoch {epoch+1}/{num_epochs}")
    print("-" * 10)

    # Iterate over data. Add tqdm here for a progress bar if desired:
    # from tqdm.notebook import tqdm
    # for batch_idx, (images, targets) in enumerate(tqdm(train_loader, desc=f"Training Epoch {epoch+1}")):
    for batch_idx, (images, targets) in enumerate(train_loader):
        # Move data to the correct device
        images = list(image.to(device) for image in images)
        # Targets is a list of dicts. Move tensors inside each dict to device.
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

        # Zero the parameter gradients
        optimizer.zero_grad()

        # Forward pass
        # Torchvision detection models return a dict of losses during training
        loss_dict = model(images, targets)

        # Sum up the losses
        losses = sum(loss for loss in loss_dict.values())

        # Check for invalid loss values (NaN or Inf)
        if not torch.isfinite(losses):
            print(f"WARNING: Non-finite loss detected: {losses.item()}. Skipping batch {batch_idx}.")
            # Optionally: print loss_dict to investigate individual losses
            # print(loss_dict)
            continue # Skip optimizer step if loss is invalid

        # Backward pass
        losses.backward()

        # Optimizer step (update weights)
        optimizer.step()

        # --- Statistics ---
        current_loss = losses.item()
        running_loss += current_loss
        epoch_train_loss += current_loss

        # Print loss statistics every N batches (e.g., every 10 batches)
        if (batch_idx + 1) % 10 == 0:
            print(f"  Batch {batch_idx+1}/{len(train_loader)} - Loss: {current_loss:.4f} (Avg: {running_loss/10:.4f})")
            # You can also print individual losses from loss_dict if needed:
            # loss_str = " ".join([f"{k}: {v.item():.4f}" for k, v in loss_dict.items()])
            # print(f"     Loss details: {loss_str}")
            running_loss = 0.0 # Reset running loss for the next N batches

    # Calculate average training loss for the epoch
    avg_epoch_train_loss = epoch_train_loss / len(train_loader)
    train_loss_history.append(avg_epoch_train_loss)
    print(f"Epoch {epoch+1} Training Loss: {avg_epoch_train_loss:.4f}")

    # Optional: Update the learning rate
    # if scheduler:
    #     scheduler.step()
    #     print(f"  LR Scheduler stepped. Current LR: {scheduler.get_last_lr()}")

    # --- Validation Phase ---
    # We'll add proper evaluation metrics (like mAP) later.
    # For now, let's just run inference to ensure the loop structure works.
    print("\nRunning Validation...")
    model.eval() # Set model to evaluation mode
    with torch.no_grad(): # Disable gradient calculations
        for images, targets in val_loader: # We still load targets for potential future evaluation
            images = list(image.to(device) for image in images)
            # In eval mode, the model expects only images and returns predictions
            outputs = model(images)
            # --- TODO: Add evaluation metric calculation here later ---
            # For now, we just run the forward pass on validation data.
            pass # Placeholder for evaluation logic

    print("Validation Phase Complete for Epoch", epoch+1)
    # --- TODO: Add logic here to save the model checkpoint, potentially based on validation metrics ---


# --- Training Complete ---
end_time = time.time()
total_time = end_time - start_time
print(f"\nTraining finished in {total_time // 60:.0f}m {total_time % 60:.0f}s")

# Optional: Plot training loss
# import matplotlib.pyplot as plt
# plt.plot(range(1, num_epochs + 1), train_loss_history, label='Training Loss')
# plt.xlabel('Epoch')
# plt.ylabel('Loss')
# plt.title('Training Loss Over Epochs')
# plt.legend()
# plt.show()